# v2 Pseudo-Label Coreset Pipeline

This notebook generates **pseudo labels** for `coreset.csv` using a local **Qwen2.5-7B-Instruct** model.

### What's fixed vs v0
| Issue | Fix |
|---|---|
| Tokenizer truncation drops system prompt | Safe context budgeting: trim only context paragraphs |
| Conflicting JSON example (Chinese think) | System + user prompts fully consistent |
| Deterministic generation warning | `do_sample=False`, no temp/top_p/top_k |
| Fragile JSON parsing | Strip fences → regex → `ast.literal_eval` fallback |
| No resume across reruns | `done_set` seeded from existing JSONL outputs |
| Errors lost | Written to `_errors.jsonl`, rerun pass writes `_fixed` / `_still_fail` |

### New v2 modules
- **Rule-based trigger extraction** (`extract_triggers`) – French trigger word list
- **Boundary check heuristic** (`boundary_check`) – modifies_scope / adds_parallel_info
- **Think template** – short structured English referencing triggers & boundary_check
- **Constraint normalisation** – whitelist tags, allowed pids, matched_paras list/str, TOPK_LINKS cap
- **Quality report** – `quality_report_v2.json` + `review_queue_v2.jsonl`

### Inputs (from `BASE_DIR`)
- `coreset.csv`
- `education_dimensions_updated.csv`
- (optional) `human_annotation.json` for few-shot examples

### Outputs (to `BASE_DIR`)
- `pseudo_labels_v2.jsonl`, `_errors`, `_fixed`, `_still_fail`, `_clean`
- `review_queue_v2.jsonl`, `quality_report_v2.json`
- (optional) `train_sft_v2.jsonl`


## Run checklist / 操作清单

### Prerequisites / 前置条件
1. Place the Qwen2.5-7B-Instruct model weights under `BASE_DIR/Qwen2.5-7B-Instruct/`.
2. Place `coreset.csv` and `education_dimensions_updated.csv` under `BASE_DIR`.
3. Confirm all paths in the **Configuration** cell (`BASE_DIR`, `OUT_MAIN`, etc.) are correct.

### First dry run on a single doc / 先用 1 个 doc 试跑
- In the **Main pseudo-labeling loop** cell, set `MAX_DOCS_DEBUG = 1` before running.
- After the run, inspect:
  - `new / skipped / errors` counts printed at the end of the loop.
  - Whether `list_item` rows have `ctx_len > 3` (confirms dynamic context is working).
  - Frequency of `Prompt too long even without context…` errors.
    If frequent, reduce `CTX_MAX_LIST` (e.g. 25 → 20) or `MAX_CTX_TOKENS` (e.g. 3800 → 3600).
- Once satisfied, set `MAX_DOCS_DEBUG = None` (disabled) and re-run for the full coreset.

### Resuming vs. full re-run / 续跑 vs. 全量重跑
- **Resume** (default): just run all cells — the pipeline skips already-processed `(doc_id, pid)` pairs.
- **Full re-run from scratch**: delete all output files before running:
  ```
  pseudo_labels_v2.jsonl
  pseudo_labels_v2_errors.jsonl
  pseudo_labels_v2_fixed.jsonl
  pseudo_labels_v2_still_fail.jsonl
  pseudo_labels_v2_clean.jsonl
  review_queue_v2.jsonl
  quality_report_v2.json
  ```
  > ⚠️ `OUT_ERRORS` (`pseudo_labels_v2_errors.jsonl`) is opened in **append** mode, so errors
  > from previous runs accumulate in it. The error-rerun pass will then reprocess all of them.
  > Delete `pseudo_labels_v2_errors.jsonl` before a fresh full run to avoid reprocessing stale entries.

### Parameter tweaks if prompt-too-long is frequent / 参数调整建议
- Reduce `CTX_MAX_LIST` from 25 to 20, **or**
- Reduce `MAX_CTX_TOKENS` from 3800 to 3600.


In [ ]:
# Install / verify dependencies  (run once, then restart kernel if needed)
import importlib, subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

for pkg, imp in [("transformers", "transformers"), ("accelerate", "accelerate"),
                 ("pandas", "pandas"), ("tqdm", "tqdm")]:
    try:
        importlib.import_module(imp)
    except ImportError:
        _pip(pkg)

print("All dependencies available.")


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
from pathlib import Path

# Root data directory (AutoDL: /root/autodl-fs, Colab: adjust as needed)
BASE_DIR = Path("/root/autodl-fs")

# Model path or Hugging Face model id
MODEL_PATH = "/root/autodl-fs/Qwen2.5-7B-Instruct"  # or "Qwen/Qwen2.5-7B-Instruct"

# I/O paths (all under BASE_DIR)
CSV_PATH       = BASE_DIR / "coreset.csv"
TAGS_CSV_PATH  = BASE_DIR / "education_dimensions_updated.csv"
HUMAN_ANN_PATH = BASE_DIR / "human_annotation.json"   # optional; used for few-shot

OUT_MAIN       = BASE_DIR / "pseudo_labels_v2.jsonl"
OUT_ERRORS     = BASE_DIR / "pseudo_labels_v2_errors.jsonl"
OUT_FIXED      = BASE_DIR / "pseudo_labels_v2_fixed.jsonl"
OUT_STILL_FAIL = BASE_DIR / "pseudo_labels_v2_still_fail.jsonl"
OUT_CLEAN      = BASE_DIR / "pseudo_labels_v2_clean.jsonl"
OUT_REVIEW     = BASE_DIR / "review_queue_v2.jsonl"
OUT_REPORT     = BASE_DIR / "quality_report_v2.json"
OUT_SFT        = BASE_DIR / "train_sft_v2.jsonl"   # optional warm-up SFT

# Generation / batching
CTX_BASE = 3
CTX_MAX_LIST = 25
CTX_MARGIN = 4

CTX_N          = 3      # context paragraphs prepended to target
MAX_NEW_TOKENS = 256    # generous but bounded
TOPK_LINKS     = 3      # max matched_paras links per paragraph
CTX_BUDGET     = 2200   # token budget for context paragraphs (safe headroom)
MAX_CTX_TOKENS = 3800   # absolute limit for whole prompt (model 4k window safety)

# Quality thresholds for clean filter
MIN_CONF_FOR_CLEAN = {"A", "B"}   # model_confidence values kept in _clean

print("Configuration loaded.")
print("  BASE_DIR :", BASE_DIR)
print("  CSV_PATH :", CSV_PATH)


In [ ]:
# ── V2 constants & French trigger word lists ──────────────────────────────────

VALID_REL  = {"contradictive", "supporting", "complemental", "modifying"}
VALID_TYPE = {"preambular", "operative"}

# French preambular / operative trigger words
TRIG_PREAMBULAR = [
    "Rappelant", "Réaffirmant", "Considérant", "Soulignant", "Notant",
    "Prenant note", "Ayant à l'esprit", "Reconnaissant", "Conscient",
    "Alarmé", "Préoccupé", "Convaincu", "Estimant", "Affirmant", "Sachant",
]
TRIG_OPERATIVE = [
    "Décide", "Invite", "Prie", "Demande", "Encourage", "Recommande",
    "Charge", "Adopte", "Appelle", "Exhorte", "Souhaite", "Autorise",
    "Établit", "Crée", "Nomme", "Proroge", "Renouvelle",
]
TRIG_ALL = sorted(set(TRIG_PREAMBULAR + TRIG_OPERATIVE), key=len, reverse=True)

# Weak-signal keyword lists for boundary_check heuristic
MODIFY_HINT = [
    "sous réserve", "à condition", "toutefois", "nonobstant", "à l'exception",
    "modifie", "remplace", "abroge", "révise", "amende", "dérogeant",
]
ADD_HINT = [
    "en outre", "par ailleurs", "de plus", "également", "notamment", "y compris",
    "ainsi que", "aussi", "et aussi",
]

# Relation priority for flattening (export)
REL_PRI = ["modifying", "contradictive", "supporting", "complemental"]

print("Constants and trigger lists loaded.")


In [ ]:
# ── Load tag whitelist from education_dimensions_updated.csv ─────────────────
import pandas as pd

def load_valid_codes(tags_csv_path: Path) -> set:
    """Read valid tag codes from the dimensions CSV."""
    if not tags_csv_path.exists():
        print(f"WARNING: {tags_csv_path} not found – using empty whitelist (all tags pass)")
        return set()
    df = pd.read_csv(tags_csv_path)
    # Try common column names
    for col in ["code", "Code", "CODE", "tag", "Tag", "id", "ID", "dimension"]:
        if col in df.columns:
            codes = set(str(x).strip() for x in df[col].dropna() if str(x).strip())
            print(f"Loaded {len(codes)} valid tag codes from column '{col}'")
            return codes
    # Fallback: use first column
    codes = set(str(x).strip() for x in df.iloc[:, 0].dropna() if str(x).strip())
    print(f"Loaded {len(codes)} valid tag codes from first column")
    return codes

VALID_CODES = load_valid_codes(TAGS_CSV_PATH)


In [ ]:
# ── Load Qwen2.5-7B-Instruct model & tokenizer ────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Loading tokenizer from: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

print(f"Loading model from:     {MODEL_PATH}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Model loaded. Device map:", {k: str(v) for k, v in model.hf_device_map.items()} \
      if hasattr(model, "hf_device_map") else "single device")


In [ ]:
# ── Rule-based helpers: trigger extraction & boundary check ──────────────────
import re

def extract_triggers(text_fr: str) -> list:
    """Extract French argument triggers from paragraph text (rule-based)."""
    t = text_fr.strip()
    hits = []
    for w in TRIG_ALL:
        if re.search(rf"(^|[\s,;:]){re.escape(w)}([\s,;:,]|$)", t, re.IGNORECASE):
            hits.append(w)
    # Deduplicate, preserve order
    seen, out = set(), []
    for x in hits:
        if x.lower() not in seen:
            out.append(x)
            seen.add(x.lower())
    return out[:5]


def infer_boundary_check(text_fr: str) -> dict:
    """Heuristic boundary check: detect scope-modification and parallel-info signals."""
    low = text_fr.lower()
    modifies = any(h in low for h in MODIFY_HINT)
    adds     = any(h in low for h in ADD_HINT)
    return {"modifies_scope": bool(modifies), "adds_parallel_info": bool(adds)}


def infer_weak_role(text_fr: str) -> str:
    """Weak prior for type: returns 'operative_anchor' or 'preambular_anchor'."""
    trig = extract_triggers(text_fr)
    if any(x in TRIG_OPERATIVE for x in trig):
        return "operative_anchor"
    if any(x in TRIG_PREAMBULAR for x in trig):
        return "preambular_anchor"
    return "unknown"


def format_think(evidence_triggers: list, boundary_check: dict) -> str:
    """Build the short structured English think string."""
    et = evidence_triggers if evidence_triggers else []
    ms = boundary_check.get("modifies_scope", False)
    ap = boundary_check.get("adds_parallel_info", False)
    return f"Triggers: {et}. Check: modifies_scope={ms}, adds_parallel_info={ap}."


print("Rule-based helpers defined.")


In [ ]:
# ── System prompt & user prompt builders ─────────────────────────────────────

SYSTEM_PROMPT = """You are an expert in argument mining for UN/UNESCO French documents.

OUTPUT RULE:
Return ONLY one valid JSON object. No markdown fences. No extra text before or after.

Required fields (all must exist):
- para_number: int
- type: "preambular" or "operative"
- tags: list of strings (from whitelist only; output [] if none fit)
- matched_paras: dict (keys = context para numbers as strings; values = one of
  "supporting" | "contradictive" | "complemental" | "modifying";
  output {} if no clear link)
- think: strictly follow this template:
  "Triggers: [list]. Check: modifies_scope=True/False, adds_parallel_info=True/False."
- evidence_triggers: list of strings
- boundary_check: {"modifies_scope": bool, "adds_parallel_info": bool}
- model_confidence: "A" | "B" | "C"
- uncertainty_flag: string ("" if none)

Relation guidance:
- modifying: the target changes scope/conditions/obligations or adds an exception to
  an earlier rule.
- complemental: the target adds parallel info/examples without changing scope.
Tie-breaker: choose modifying ONLY when scope/conditions explicitly change.
"""


def build_user_prompt(doc_id: str, pid, text_fr: str, prev_rows: list,
                      valid_codes: set, weak_role_hint: str = "unknown") -> str:
    """Build the user prompt.  prev_rows are the context paragraphs (already trimmed)."""
    ctx_lines = [f"<P{int(r['pid'])}> {str(r['text_fr'])}" for r in prev_rows]
    code_str   = ", ".join(sorted(valid_codes)) if valid_codes else "(all tags allowed)"
    hint_str   = weak_role_hint if weak_role_hint in (
        "operative_anchor", "preambular_anchor") else "unknown"

    return (
        f"Task: annotate the TARGET paragraph using the provided CONTEXT paragraphs.\n\n"
        f"doc_id: {doc_id}\n"
        f"weak_role_hint: {hint_str}\n\n"
        f"CONTEXT:\n"
        f"{chr(10).join(ctx_lines) if ctx_lines else '(no context)'}\n\n"
        f"TARGET:\n"
        f"<P{int(pid)}> {text_fr} </P{int(pid)}>\n\n"
        f"TAG WHITELIST:\n[{code_str}]\n\n"
        f"Return ONLY one JSON object."
    )


print("Prompt builders defined.")


In [ ]:
# ── Safe context budgeting (prevents system-prompt truncation) ────────────────

def build_chat_messages(doc_id: str, pid, text_fr: str, prev_rows: list,
                        valid_codes: set, weak_role_hint: str = "unknown"):
    """Build chat messages list for apply_chat_template."""
    user_prompt = build_user_prompt(doc_id, pid, text_fr, prev_rows,
                                    valid_codes, weak_role_hint)
    return [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_prompt},
    ]


def safe_build_inputs(doc_id: str, pid, text_fr: str, ctx_rows: list,
                      valid_codes: set, weak_role_hint: str = "unknown",
                      ctx_budget: int = CTX_BUDGET):
    """
    Build tokenized inputs with safe context budgeting.

    Strategy:
    1. Count tokens used by system + target-only user prompt.
    2. Any remaining budget up to ctx_budget is given to context paragraphs.
    3. Context paragraphs are trimmed from the *oldest* (furthest away) first
       until they fit, never truncating the system prompt or target.
    """
    # Step 1 – compute fixed cost (system + target only, no context)
    no_ctx_msgs = build_chat_messages(doc_id, pid, text_fr, [],
                                      valid_codes, weak_role_hint)
    no_ctx_ids  = tokenizer.apply_chat_template(
        no_ctx_msgs, tokenize=True, add_generation_prompt=True)
    fixed_cost  = len(no_ctx_ids)

    # Step 2 – how many tokens can context use?
    available   = max(0, ctx_budget - fixed_cost)

    # Step 3 – greedily add context paras from nearest → furthest
    kept_rows   = []
    used_tokens = 0
    for row in reversed(ctx_rows):          # nearest first
        ctx_text  = f"<P{int(row['pid'])}> {str(row['text_fr'])}"
        ctx_toks  = len(tokenizer.encode(ctx_text, add_special_tokens=False))
        if used_tokens + ctx_toks > available:
            break
        kept_rows.insert(0, row)            # restore chronological order
        used_tokens += ctx_toks

    msgs   = build_chat_messages(doc_id, pid, text_fr, kept_rows,
                                 valid_codes, weak_role_hint)
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )

    # Step 4 – hard cap total prompt length to MAX_CTX_TOKENS (drop oldest ctx only)
    # This never truncates system/target; it only removes context rows.
    while inputs.shape[-1] > MAX_CTX_TOKENS and kept_rows:
        kept_rows = kept_rows[1:]  # drop oldest context paragraph
        msgs = build_chat_messages(doc_id, pid, text_fr, kept_rows,
                                   valid_codes, weak_role_hint)
        inputs = tokenizer.apply_chat_template(
            msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        )

    # Extreme case: even with no context, still too long (target itself huge)
    if inputs.shape[-1] > MAX_CTX_TOKENS:
        raise ValueError(
            f"Prompt too long even without context: doc_id={doc_id} pid={pid} "
            f"{int(inputs.shape[-1])} > {MAX_CTX_TOKENS}"
        )

    return inputs, kept_rows


print("Safe context budgeting defined.")


In [ ]:
# ── Dynamic context expansion for list_item (enum-based) ──────────────────────
import re
import pandas as pd

ENUM_RE = re.compile(r"^\s*((\d+(?:\.\d+)+)|(\d+)|([a-zA-Z])|([ivxlcdm]+))\s*[\)\.\:]")
OPER_GUIDE_RE = re.compile(r"^(Demande|Invite|Prie|Recommande|Décide|Encourage|Engage|Appelle)\b", re.I)

def roman_to_int(s):
    s = s.lower()
    vals = {'i':1,'v':5,'x':10,'l':50,'c':100,'d':500,'m':1000}
    total, prev = 0, 0
    for ch in reversed(s):
        v = vals.get(ch, 0)
        if v < prev: total -= v
        else: total += v; prev = v
    return total if total > 0 else None

def enum_index(text):
    m = ENUM_RE.match(text or "")
    if not m:
        return None
    token = m.group(1)
    if not token:
        return None
    if re.fullmatch(r"\d+(\.\d+)+", token):
        return int(token.split(".")[-1])
    if token.isdigit():
        return int(token)
    if re.fullmatch(r"[a-zA-Z]", token):
        return ord(token.lower()) - ord("a") + 1
    if re.fullmatch(r"[ivxlcdm]+", token.lower()):
        return roman_to_int(token)
    return None

def looks_like_guide(text: str) -> bool:
    if not isinstance(text, str):
        return False
    s = text.strip()
    return s.endswith(":") or bool(OPER_GUIDE_RE.match(s))

def get_ctx_len(row, base_ctx=3, ctx_max=25, margin=4):
    """Dynamic ctx length for list_item; otherwise base_ctx."""
    typ = row.get("type", "")
    parent = row.get("parent", None)
    if typ == "list_item" and pd.notna(parent) and str(parent).strip() != "":
        ei = enum_index(str(row.get("text_fr", "")))
        if ei is None:
            # no enum marker → moderate expansion
            return min(ctx_max, max(base_ctx, base_ctx + 8))
        return min(ctx_max, max(base_ctx, ei + margin))
    return base_ctx

def build_dynamic_ctx(doc_rows, i, base_ctx=3, ctx_max=25, margin=4):
    """
    doc_rows: list[dict] for one doc, in pid order
    i: index of target row in doc_rows
    Return context rows (previous only) with dynamic expansion and guide-search.
    """
    row = doc_rows[i]
    ctx_len = get_ctx_len(row, base_ctx=base_ctx, ctx_max=ctx_max, margin=margin)

    # expand until guide appears (list_item only)
    if row.get("type", "") == "list_item":
        while True:
            start = max(0, i - ctx_len)
            ctx = doc_rows[start:i]
            if any(looks_like_guide(r.get("text_fr","")) for r in ctx):
                break
            if ctx_len >= ctx_max or start == 0:
                break
            ctx_len = min(ctx_max, ctx_len + 4)
        return doc_rows[max(0, i - ctx_len):i]

    return doc_rows[max(0, i - ctx_len):i]

In [ ]:
# ── Robust JSON extraction / parsing ──────────────────────────────────────────
import ast, json, re as _re

def strip_code_fences(text: str) -> str:
    """Remove markdown code fences like ```json ... ``` or ``` ... ```."""
    text = _re.sub(r"```(?:json)?\s*", "", text)
    text = text.replace("```", "")
    return text.strip()


def extract_json_object(text: str) -> dict | None:
    """
    Try to extract the first JSON object from text.
    Steps:
      1. Strip code fences.
      2. regex scan for { ... } (outermost braces).
      3. json.loads.
      4. ast.literal_eval as last resort.
    Returns None on failure.
    """
    text = strip_code_fences(text)

    # Find the first { and last }
    start = text.find("{")
    if start == -1:
        return None
    # Walk forward to find balanced closing brace
    depth, end = 0, -1
    for i, ch in enumerate(text[start:], start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                end = i
                break
    if end == -1:
        return None

    candidate = text[start:end + 1]

    # Attempt 1: json.loads
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        pass

    # Attempt 2: ast.literal_eval (handles single-quoted dicts)
    try:
        result = ast.literal_eval(candidate)
        if isinstance(result, dict):
            return result
    except Exception:
        pass

    return None


print("JSON extraction helpers defined.")


In [ ]:
# ── normalize_pred: constraint checking & normalisation ──────────────────────

def flatten_rel_list(rels: list) -> str | None:
    """Given a list of relations, pick the highest-priority one."""
    for r in REL_PRI:
        if r in rels:
            return r
    return rels[0] if rels else None


def normalize_pred(obj: dict, pid, allowed_ctx_pids: set,
                   text_fr: str, weak_role_hint: str = "unknown") -> dict:
    """
    Validate and normalise a raw model output dict.

    - Enforces type from weak_role_hint / trigger prior.
    - Filters tags against VALID_CODES whitelist.
    - Cleans matched_paras: only allowed pids, valid relations, list or str input,
      de-duplicates, caps at TOPK_LINKS.
    - Fills evidence_triggers and boundary_check from rules if model omits them.
    - Forces think template (English, consistent format).
    """
    default_type = "operative" if weak_role_hint == "operative_anchor" else "preambular"

    out = {
        "para_number":      int(pid),
        "type":             default_type,
        "tags":             [],
        "matched_paras":    {},
        "think":            "",
        "evidence_triggers": [],
        "boundary_check":   {"modifies_scope": False, "adds_parallel_info": False},
        "model_confidence": "C",
        "uncertainty_flag": "",
    }

    # ── type ──────────────────────────────────────────────────────────────────
    para_type = str(obj.get("type", default_type)).strip().lower()
    if para_type not in VALID_TYPE:
        para_type = default_type
    # Rule override: operative trigger words in text → force operative
    trig = extract_triggers(text_fr)
    if any(x in TRIG_OPERATIVE for x in trig):
        para_type = "operative"
    out["type"] = para_type

    # ── tags ──────────────────────────────────────────────────────────────────
    raw_tags = obj.get("tags", [])
    if isinstance(raw_tags, str):
        raw_tags = [raw_tags]
    if not isinstance(raw_tags, list):
        raw_tags = []
    fixed_tags, invalid_tags = [], []
    for t in raw_tags:
        t = str(t).strip()
        if not t:
            continue
        if (not VALID_CODES) or (t in VALID_CODES):
            fixed_tags.append(t)
        else:
            invalid_tags.append(t)
    out["tags"]          = list(dict.fromkeys(fixed_tags))   # deduplicated
    out["_invalid_tags"] = invalid_tags

    # ── matched_paras ─────────────────────────────────────────────────────────
    mp = obj.get("matched_paras", {})
    if not isinstance(mp, dict):
        mp = {}
    clean_mp = {}
    for k, v in mp.items():
        k_str = str(k).strip().lstrip("P").lstrip("p")
        if not k_str.isdigit():
            continue
        k_int = int(k_str)
        if k_int not in allowed_ctx_pids:
            continue
        # Accept both string and list values
        if isinstance(v, str):
            rels = [v.strip().lower()]
        elif isinstance(v, list):
            rels = [str(x).strip().lower() for x in v]
        else:
            rels = []
        rels = [r for r in rels if r in VALID_REL]
        rels = list(dict.fromkeys(rels))   # deduplicate, preserve order
        if rels:
            best = flatten_rel_list(rels)
            if best:
                clean_mp[str(k_int)] = best   # store as string (submission-safe)

    # Cap at TOPK_LINKS – prefer nearest paragraphs
    if len(clean_mp) > TOPK_LINKS:
        keys = sorted(clean_mp, key=lambda x: abs(int(x) - int(pid)))
        clean_mp = {k: clean_mp[k] for k in keys[:TOPK_LINKS]}
    out["matched_paras"] = clean_mp

    # ── evidence_triggers (rule-first, model-second) ──────────────────────────
    model_et = obj.get("evidence_triggers", [])
    if not isinstance(model_et, list) or len(model_et) == 0:
        model_et = trig
    out["evidence_triggers"] = model_et

    # ── boundary_check (merge rule + model) ───────────────────────────────────
    rule_bc  = infer_boundary_check(text_fr)
    model_bc = obj.get("boundary_check", {})
    if not isinstance(model_bc, dict):
        model_bc = {}
    out["boundary_check"] = {
        "modifies_scope":    bool(model_bc.get("modifies_scope",
                                               rule_bc["modifies_scope"])),
        "adds_parallel_info": bool(model_bc.get("adds_parallel_info",
                                                rule_bc["adds_parallel_info"])),
    }

    # ── think (always enforce template) ──────────────────────────────────────
    out["think"] = format_think(out["evidence_triggers"], out["boundary_check"])[:300]

    # ── confidence / uncertainty ──────────────────────────────────────────────
    conf = str(obj.get("model_confidence", "C")).strip().upper()
    out["model_confidence"]  = conf if conf in {"A", "B", "C"} else "C"
    out["uncertainty_flag"]  = str(obj.get("uncertainty_flag", "")).strip()[:200]

    return out


print("normalize_pred defined.")


In [ ]:
# ── Rule linker: fill matched_paras for list_item (guide + prev_sibling) ───────

def rule_fill_links(pid, row, ctx_rows, topk=TOPK_LINKS):
    """
    Return a matched_paras dict (string relation values) using only ctx_rows.
    Strategy C:
      - link to nearest guide paragraph in ctx (if exists)
      - link to previous sibling list_item with same parent in ctx (if exists)
    """
    out = {}

    if not ctx_rows:
        return out

    parent = row.get("parent", None)
    is_list = (row.get("type", "") == "list_item") and (parent is not None) and str(parent).strip() != ""

    if not is_list:
        return out

    # 1) nearest guide in context (search backwards)
    guide_pid = None
    for r in reversed(ctx_rows):
        if looks_like_guide(str(r.get("text_fr", ""))):
            guide_pid = int(r["pid"])
            break
    if guide_pid is not None:
        out[str(guide_pid)] = "complemental"

    # 2) previous sibling with same parent (search backwards)
    sib_pid = None
    for r in reversed(ctx_rows):
        if r.get("type", "") == "list_item" and str(r.get("parent", "")).strip() == str(parent).strip():
            sib_pid = int(r["pid"])
            break
    if sib_pid is not None:
        # avoid duplicate key if guide_pid == sib_pid (rare but safe)
        if str(sib_pid) not in out:
            out[str(sib_pid)] = "complemental"

    # Cap to topk, prefer nearest pid
    if len(out) > topk:
        keys = sorted(out.keys(), key=lambda x: abs(int(x) - int(pid)))
        out = {k: out[k] for k in keys[:topk]}

    return out

In [ ]:
# ── Core inference: generate annotation for one paragraph ─────────────────────
import torch

@torch.inference_mode()
def generate_annotation(doc_id: str, pid, text_fr: str, ctx_rows: list,
                         valid_codes: set, device="cuda",
                         row_meta: dict | None = None) -> dict:
    """
    Run the model and return a normalised annotation dict.
    Raises ValueError if JSON cannot be extracted.
    """
    weak_hint = infer_weak_role(text_fr)
    allowed   = {int(r["pid"]) for r in ctx_rows}

    inputs, kept_rows = safe_build_inputs(
        doc_id, pid, text_fr, ctx_rows, valid_codes, weak_hint)
    inputs = inputs.to(device)

    # Deterministic generation – do NOT pass temperature / top_p / top_k
    out_ids = model.generate(
        inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    # Decode only the newly generated tokens
    new_tokens = out_ids[0, inputs.shape[-1]:]
    raw_text   = tokenizer.decode(new_tokens, skip_special_tokens=True)

    obj = extract_json_object(raw_text)
    if obj is None:
        raise ValueError(f"No JSON found in model output: {raw_text[:300]!r}")

    row_meta = row_meta or {}
    annotation = normalize_pred(obj, pid, allowed, text_fr, weak_hint)
    # Rule-based fallback for list_item when model gives no links
    if (not annotation.get("matched_paras")) and ctx_rows:
        annotation["matched_paras"] = rule_fill_links(pid, row_meta, ctx_rows, topk=TOPK_LINKS)
    return annotation, raw_text


print("generate_annotation defined.")


In [ ]:
# ── Load coreset.csv ──────────────────────────────────────────────────────────
import pandas as pd

df = pd.read_csv(CSV_PATH)
# Normalise column names to lowercase
df.columns = [c.strip().lower() for c in df.columns]
print("Columns:", list(df.columns))
print("Shape  :", df.shape)
print(df.head(3))


In [ ]:
# ── Resume support: build done_set from existing outputs ──────────────────────

def load_done_set(out_path: Path) -> set:
    """Read existing JSONL and return set of (doc_id, str(pid)) already done."""
    done = set()
    if not out_path.exists():
        return done
    with open(out_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                done.add((str(rec["doc_id"]), str(rec["pid"])))
            except Exception:
                pass
    return done

done_set = load_done_set(OUT_MAIN)
print(f"Resuming: {len(done_set)} (doc_id, pid) pairs already done.")


In [ ]:
# ── Main pseudo-labeling loop ──────────────────────────────────────────────────
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

# Ensure coreset is sorted
df = df.sort_values(["doc_id", "pid"]).reset_index(drop=True)

out_main_f   = open(OUT_MAIN,   "a", encoding="utf-8")
# OUT_ERRORS is opened in append mode — errors accumulate across runs.
# ⚠️  Delete pseudo_labels_v2_errors.jsonl before a fresh full run to avoid
#    reprocessing stale errors in the rerun-errors pass.
out_errors_f = open(OUT_ERRORS, "a", encoding="utf-8")

# ── Debug gate: set MAX_DOCS_DEBUG = N to process only the first N docs,
# then break early (useful for a quick dry run). Set to None to disable.
MAX_DOCS_DEBUG = None  # e.g. MAX_DOCS_DEBUG = 1 for a single-doc dry run

n_done = n_skip = n_err = 0
_doc_count = 0

for doc_id, doc_df in tqdm(df.groupby("doc_id", sort=False), desc="docs"):
    _doc_count += 1
    if MAX_DOCS_DEBUG is not None and _doc_count > MAX_DOCS_DEBUG:
        break
    doc_rows = doc_df.to_dict("records")
    # Build pid → row index map for quick context lookup
    pid_list = [r["pid"] for r in doc_rows]

    for i, row in enumerate(doc_rows):
        pid      = row["pid"]
        text_fr  = str(row.get("text_fr", row.get("para", "")))
        key      = (str(doc_id), str(pid))

        if key in done_set:
            n_skip += 1
            continue

        # Context: up to CTX_N preceding paragraphs
        ctx_rows = build_dynamic_ctx(doc_rows, i, base_ctx=CTX_BASE, ctx_max=CTX_MAX_LIST, margin=CTX_MARGIN)

        try:
            annotation, raw_text = generate_annotation(
                doc_id, pid, text_fr, ctx_rows, VALID_CODES, device,
                row_meta=row)

            record = {
                "doc_id":     str(doc_id),
                "pid":        str(pid),
                "text_fr":    text_fr,
                "annotation": annotation,
            }
            out_main_f.write(json.dumps(record, ensure_ascii=False) + "\n")
            out_main_f.flush()
            done_set.add(key)
            n_done += 1

        except Exception as e:
            err_record = {
                "doc_id":  str(doc_id),
                "pid":     str(pid),
                "text_fr": text_fr,
                "error":   str(e),
            }
            out_errors_f.write(json.dumps(err_record, ensure_ascii=False) + "\n")
            out_errors_f.flush()
            n_err += 1

out_main_f.close()
out_errors_f.close()

print(f"\nDone. new={n_done}  skipped={n_skip}  errors={n_err}")
print(f"  Main output   : {OUT_MAIN}")
print(f"  Errors        : {OUT_ERRORS}")


In [ ]:
# ── Error rerun pass ───────────────────────────────────────────────────────────
# Reads pseudo_labels_v2_errors.jsonl, rebuilds context from coreset.csv,
# writes _fixed.jsonl and _still_fail.jsonl

if not OUT_ERRORS.exists() or OUT_ERRORS.stat().st_size == 0:
    print("No errors to rerun.")
else:
    # Build a lookup: (doc_id, pid) → row index in df
    df_lookup = {(str(r["doc_id"]), str(r["pid"])): i
                 for i, r in df.iterrows()}

    with (open(OUT_ERRORS, "r", encoding="utf-8") as ef,
          open(OUT_FIXED,      "w", encoding="utf-8") as ff,
          open(OUT_STILL_FAIL, "w", encoding="utf-8") as sf):

        for line in tqdm(ef, desc="rerun errors"):
            line = line.strip()
            if not line:
                continue
            err_rec = json.loads(line)
            doc_id  = str(err_rec["doc_id"])
            pid     = str(err_rec["pid"])

            idx = df_lookup.get((doc_id, pid))
            if idx is None:
                sf.write(json.dumps(err_rec, ensure_ascii=False) + "\n")
                continue

            row     = df.iloc[idx].to_dict()
            text_fr = str(row.get("text_fr", row.get("para", "")))

            # Rebuild context via build_dynamic_ctx over the full doc
            doc_all = df[df["doc_id"].astype(str) == doc_id].sort_values("pid").to_dict("records")
            # find index i in doc_all where pid matches
            i = next((j for j,r in enumerate(doc_all) if str(r["pid"]) == str(pid)), None)
            if i is None:
                ctx_rows = []
            else:
                ctx_rows = build_dynamic_ctx(doc_all, i, base_ctx=CTX_BASE, ctx_max=CTX_MAX_LIST, margin=CTX_MARGIN)

            try:
                annotation, raw_text = generate_annotation(
                    doc_id, pid, text_fr, ctx_rows, VALID_CODES, device,
                    row_meta=row)
                record = {
                    "doc_id":     doc_id,
                    "pid":        pid,
                    "text_fr":    text_fr,
                    "annotation": annotation,
                }
                ff.write(json.dumps(record, ensure_ascii=False) + "\n")
            except Exception as e2:
                err_rec["rerun_error"] = str(e2)
                sf.write(json.dumps(err_rec, ensure_ascii=False) + "\n")

    print(f"Rerun done. Fixed: {OUT_FIXED}  Still failing: {OUT_STILL_FAIL}")


In [ ]:
# ── Produce pseudo_labels_v2_clean.jsonl ─────────────────────────────────────
# Merges main + fixed, deduplicates, keeps only high-confidence annotations.

merged = {}   # (doc_id, pid) → record

for src_path in [OUT_MAIN, OUT_FIXED]:
    if not src_path.exists():
        continue
    with open(src_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            key = (str(rec["doc_id"]), str(rec["pid"]))
            merged[key] = rec    # later file wins (fixed overrides main)

n_clean = 0
with open(OUT_CLEAN, "w", encoding="utf-8") as cf:
    for rec in sorted(merged.values(), key=lambda r: (str(r["doc_id"]), float(r["pid"]))):
        ann = rec.get("annotation", {})
        if ann.get("model_confidence", "C") in MIN_CONF_FOR_CLEAN:
            cf.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n_clean += 1

print(f"Clean file written: {OUT_CLEAN}  ({n_clean} records)")


In [ ]:
# ── Boundary consistency check → review_queue_v2.jsonl ───────────────────────

def boundary_consistency_issue(ann: dict) -> str | None:
    """
    Returns a description of the consistency issue, or None if clean.
    """
    bc  = ann.get("boundary_check", {})
    mp  = ann.get("matched_paras", {})
    rels = {r for v in mp.values() for r in (v if isinstance(v, list) else [v])}

    if bc.get("modifies_scope") and "modifying" not in rels and mp:
        return "modifies_scope=True but no modifying relation"
    if (not bc.get("modifies_scope")) and "modifying" in rels:
        return "modifying relation but modifies_scope=False"
    # Same pid has both complemental and modifying → conflict
    for pid_k, v in mp.items():
        v_list = v if isinstance(v, list) else [v]
        if "complemental" in v_list and "modifying" in v_list:
            return f"pid {pid_k} has both complemental and modifying"
    return None

review_count = 0
with (open(OUT_CLEAN,  "r", encoding="utf-8") as cf,
      open(OUT_REVIEW, "w", encoding="utf-8") as rf):
    for line in cf:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        issue = boundary_consistency_issue(rec.get("annotation", {}))
        if issue:
            rf.write(json.dumps({"issue": issue, **rec}, ensure_ascii=False) + "\n")
            review_count += 1

print(f"Review queue: {review_count} items → {OUT_REVIEW}")


In [ ]:
# ── Quality report: quality_report_v2.json ───────────────────────────────────
from collections import Counter

type_dist   = Counter()
tag_dist    = Counter()
rel_dist    = Counter()
n_invalid   = 0
conf_dist   = Counter()
n_total     = 0
covered_docs = set()
covered_pids = set()

all_pids_in_csv = set(zip(df["doc_id"].astype(str), df["pid"].astype(str)))

with open(OUT_CLEAN, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        ann = rec.get("annotation", {})
        n_total += 1
        covered_docs.add(str(rec["doc_id"]))
        covered_pids.add((str(rec["doc_id"]), str(rec["pid"])))

        type_dist[ann.get("type", "unknown")] += 1
        for tag in ann.get("tags", []):
            tag_dist[tag] += 1
        n_invalid += len(ann.get("_invalid_tags", []))
        for v in ann.get("matched_paras", {}).values():
            rels = v if isinstance(v, list) else [v]
            for r in rels:
                rel_dist[r] += 1
        conf_dist[ann.get("model_confidence", "C")] += 1

missing_pids = all_pids_in_csv - covered_pids

report = {
    "n_total_annotations":      n_total,
    "n_docs_covered":           len(covered_docs),
    "n_total_docs_in_csv":      df["doc_id"].nunique(),
    "n_paragraphs_in_csv":      len(all_pids_in_csv),
    "n_paragraphs_covered":     len(covered_pids),
    "n_paragraphs_missing":     len(missing_pids),
    "type_distribution":        dict(type_dist),
    "tag_distribution":         dict(tag_dist.most_common(30)),
    "relation_distribution":    dict(rel_dist),
    "invalid_tags_total":       n_invalid,
    "confidence_distribution":  dict(conf_dist),
    "review_queue_count":       review_count,
}

with open(OUT_REPORT, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Quality report written to", OUT_REPORT)
for k, v in report.items():
    if k not in ("tag_distribution",):
        print(f"  {k}: {v}")


In [ ]:
# ── (Optional) Export warm-up SFT data: train_sft_v2.jsonl ──────────────────
# Only exports high-confidence (A/B) records as chat-format training examples.

EXPORT_SFT = True   # set to False to skip

if EXPORT_SFT:
    n_sft = 0
    with (open(OUT_CLEAN, "r", encoding="utf-8") as cf,
          open(OUT_SFT,   "w", encoding="utf-8") as sf):
        for line in cf:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            ann = rec.get("annotation", {})
            if ann.get("model_confidence", "C") not in {"A", "B"}:
                continue

            doc_id  = str(rec["doc_id"])
            pid     = rec["pid"]
            text_fr = rec["text_fr"]

            # Build a minimal chat example (system + user + assistant)
            # Assistant output is the cleaned annotation JSON
            ann_clean = {k: v for k, v in ann.items()
                         if not k.startswith("_")}  # strip _invalid_tags etc.
            assistant_content = json.dumps(ann_clean, ensure_ascii=False)

            sft_rec = {
                "messages": [
                    {"role": "system",    "content": SYSTEM_PROMPT},
                    {"role": "user",      "content":
                        build_user_prompt(doc_id, pid, text_fr, [], VALID_CODES)},
                    {"role": "assistant", "content": assistant_content},
                ]
            }
            sf.write(json.dumps(sft_rec, ensure_ascii=False) + "\n")
            n_sft += 1

    print(f"SFT file written: {OUT_SFT}  ({n_sft} examples)")
else:
    print("SFT export skipped.")


In [ ]:
# ── Sanity check: sample a few records ───────────────────────────────────────
import random

print("=== Sample records from pseudo_labels_v2_clean.jsonl ===")
records = []
with open(OUT_CLEAN, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

if records:
    samples = random.sample(records, min(3, len(records)))
    for s in samples:
        print(f"\ndoc_id={s['doc_id']}  pid={s['pid']}")
        ann = s["annotation"]
        print(f"  type={ann['type']}  conf={ann['model_confidence']}")
        print(f"  tags={ann['tags']}")
        print(f"  matched_paras={ann['matched_paras']}")
        print(f"  think: {ann['think']}")
else:
    print("(no records yet – run the labeling loop first)")
